# 13 - Generate Submission
Load a checkpoint and generate plain and reranked submission files.

In [7]:
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data as data
import src.inference as inference
from src.models import load_arcface_model_from_checkpoint
from src.training import compute_map_from_embeddings
from src.utils import get_device, set_seed

load_dotenv(project_root / ".env")

RANDOM_SEED = 42
set_seed(RANDOM_SEED)

device = get_device()
device

device(type='cuda')

## Config

In [8]:
config = {
    # "checkpoint_path": Path("../checkpoints/e10_hyperparamter_search/best_eva_unfrozen_rs_01_hlr3e-04_blr1e-05_wd5e-04_do0.2_aug1_bs32.pth"),
    "checkpoint_path": Path("../checkpoints/e10_hyperparamter_search/best_eva_unfrozen_rs_06_hlr3e-05_blr1e-05_wd5e-04_do0.3_aug1_bs32.pth"),
    "output_dir": Path("../submissions/e10_hyperparamter_search"),
    "compute_validation_metrics": True,
    "generate_plain": True,
    "generate_rerank": True,
    "num_workers": 0,
    "batch_size": None,
    "seed": RANDOM_SEED,
}

config["output_dir"].mkdir(parents=True, exist_ok=True)
config

{'checkpoint_path': PosixPath('../checkpoints/e10_hyperparamter_search/best_eva_unfrozen_rs_06_hlr3e-05_blr1e-05_wd5e-04_do0.3_aug1_bs32.pth'),
 'output_dir': PosixPath('../submissions/e10_hyperparamter_search'),
 'compute_validation_metrics': True,
 'generate_plain': True,
 'generate_rerank': True,
 'num_workers': 0,
 'batch_size': None,
 'seed': 42}

## Generate Submission

In [9]:
checkpoint_path = config["checkpoint_path"]
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

model, checkpoint, checkpoint_config = load_arcface_model_from_checkpoint(checkpoint_path, device)

eval_frames = data.build_eval_frames_from_config(checkpoint_config)
data_dir = Path(checkpoint_config["data_dir"])
pairs_df = data.load_test_pairs_df(data_dir)
unique_images = sorted(set(pairs_df["query_image"].unique()) | set(pairs_df["gallery_image"].unique()))
test_df = pd.DataFrame({"filename": unique_images})

input_size = int(checkpoint_config["input_size"])
batch_size = config["batch_size"] or int(checkpoint_config.get("batch_size", 32))
num_workers = int(config["num_workers"])
seed = int(checkpoint_config.get("seed", config["seed"]))
rerank_config = checkpoint_config.get("rerank", {"enabled": False, "k1": 20, "k2": 6, "lambda_value": 0.3})

if config["compute_validation_metrics"]:
    val_transform = data.build_transforms(model.backbone, input_size)
    val_loader = data.create_eval_loader(
        eval_frames["val_df"],
        eval_frames["data_dir"] / "train" / "train",
        val_transform,
        batch_size=batch_size,
        num_workers=num_workers,
        is_test=False,
        seed=seed,
    )
    val_embeddings, val_labels = inference.extract_embeddings_with_labels(model, val_loader, device)
    val_map = compute_map_from_embeddings(val_embeddings, val_labels)
    val_map_rerank = compute_map_from_embeddings(
        val_embeddings,
        val_labels,
        use_rerank=rerank_config.get("enabled", False),
        k1=rerank_config.get("k1", 20),
        k2=rerank_config.get("k2", 6),
        lambda_value=rerank_config.get("lambda_value", 0.3),
    )
    print(f"Validation mAP: {val_map:.4f}")
    print(f"Validation mAP rerank: {val_map_rerank:.4f}")

transform = data.build_transforms(model.backbone, input_size)
test_loader = data.create_eval_loader(
    test_df,
    data_dir / "test" / "test",
    transform,
    batch_size=batch_size,
    num_workers=num_workers,
    is_test=True,
    seed=seed,
)

plain_output = config["output_dir"] / f"{checkpoint_path.stem}_submission.csv"
rerank_output = config["output_dir"] / f"{checkpoint_path.stem}_submission_rerank.csv"

if config["generate_plain"]:
    inference.create_submission_model(
        model,
        device,
        pairs_df,
        test_loader,
        output_path=plain_output,
        use_rerank=False,
    )
    print(f"Plain submission written to: {plain_output}")

if config["generate_rerank"]:
    inference.create_submission_model(
        model,
        device,
        pairs_df,
        test_loader,
        output_path=rerank_output,
        use_rerank=rerank_config.get("enabled", False),
        k1=rerank_config.get("k1", 20),
        k2=rerank_config.get("k2", 6),
        lambda_value=rerank_config.get("lambda_value", 0.3),
    )
    print(f"Rerank submission written to: {rerank_output}")

print("Done")


Embedding:   0%|          | 0/12 [00:00<?, ?it/s]

Validation mAP: 0.9098
Validation mAP rerank: 0.9096


Embedding:   0%|          | 0/12 [00:00<?, ?it/s]

Building submission scores:   0%|          | 0/137270 [00:00<?, ?it/s]

Plain submission written to: ../submissions/e10_hyperparamter_search/best_eva_unfrozen_rs_06_hlr3e-05_blr1e-05_wd5e-04_do0.3_aug1_bs32_submission.csv


Embedding:   0%|          | 0/12 [00:00<?, ?it/s]

Building submission scores:   0%|          | 0/137270 [00:00<?, ?it/s]

Rerank submission written to: ../submissions/e10_hyperparamter_search/best_eva_unfrozen_rs_06_hlr3e-05_blr1e-05_wd5e-04_do0.3_aug1_bs32_submission_rerank.csv
Done
